In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
import os

def parse_spline_block(block_lines):
    points = []
    for line in block_lines:
        tokens = line.strip().split()
        if len(tokens) >= 4:
            x, y, frame, orientation = map(float, tokens[:4])
            points.append((frame, x, y))
    return sorted(points)

def interpolate_trajectory(points, step=1):
    frames, xs, ys = zip(*points)
    f_x = interp1d(frames, xs, kind='linear', fill_value='extrapolate')
    f_y = interp1d(frames, ys, kind='linear', fill_value='extrapolate')

    new_frames = np.arange(frames[0], frames[-1] + 1, step)
    new_xs = f_x(new_frames)
    new_ys = f_y(new_frames)

    return list(zip(new_frames, new_xs, new_ys))

def convert_file_to_dataframe(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()
        #print(lines)

    i = 0
    person_id = 0
    data = []

    while i < len(lines):
        line = lines[i].strip().lower()
        if "num of control points" in line:
            try:
                num_points = int(line.split()[0])
                block = lines[i + 1: i + 1 + num_points]
                raw_points = parse_spline_block(block)
                interp_points = interpolate_trajectory(raw_points)
                for frame, x, y in interp_points:
                    #print(frame)
                    data.append([int(frame), person_id, x, y])
                person_id += 1
                i += 1 + num_points
            except Exception as e:
                print(f"Failed at line {i}: {e}")
                i += 1
        else:
            i += 1

    #print(data)
    return pd.DataFrame(data, columns=['frame_id', 'person_id', 'x', 'y'])


In [2]:
# --------- Configuration ---------
num = 2  # Change this to select a different Zara dataset
vsp_file = f"./crowds/data/crowds_zara0{num}.vsp"
df = convert_file_to_dataframe(vsp_file)
print(f"Saving {len(df)} rows to converted_zara_{num}.csv")
df.to_csv(f"converted_zara_{num}.csv", index=False)

Saving 95342 rows to converted_zara_2.csv


In [3]:
print(df.head())

   frame_id  person_id           x          y
0         6          0  358.000000 -66.000000
1         7          0  355.906977 -65.906977
2         8          0  353.813953 -65.813953
3         9          0  351.720930 -65.720930
4        10          0  349.627907 -65.627907
